# Day 6 目标：实现 Document Loader + Metadata Schema

今天只做一件事：

把 Day 5 写好的 data/papers、data/experiments、data/datasets 里的 markdown 文档，加载成 LangChain Document，并自动附带 metadata。

今天先不做：

Chroma
Embedding
向量索引
Retriever Routing
LangGraph 节点接入

Day 6 的核心是理解：

Markdown 文件
↓
读取正文
↓
解析基本信息
↓
生成 Document(page_content=..., metadata=...)
↓
后续给 Chroma / Retriever 使用

# 一、Day 6 最终目录结构

今天新增这个目录：

F:\ResearchAgent
├── src
│   └── research_agent
│       ├── rag
│       │   ├── __init__.py
│       │   ├── schemas.py
│       │   └── loaders.py
│       │
│       └── graph
│           ├── state.py
│           ├── nodes.py
│           ├── router.py
│           ├── workflow.py
│           └── llm_classifier.py
│
└── scripts
    └── test_load_docs.py

今天主要写 3 个文件：

| 文件                          | 作用                        |
| --------------------------- | ------------------------- |
| `schemas.py`                | 定义资料类型和 metadata 字段       |
| `loaders.py`                | 读取 markdown，生成 `Document` |
| `scripts/test_load_docs.py` | 测试 loader 是否正常工作          |


# 二、先安装依赖

你需要确认已经安装 LangChain 的核心包。

在终端运行：

cd F:\ResearchAgent
.\.conda\python.exe -m pip install langchain-core langchain

如果你之前已经装过，也可以跳过。

# 三、创建目录和文件

在 PowerShell 运行：

cd F:\ResearchAgent

mkdir src\research_agent\rag
New-Item src\research_agent\rag\__init__.py
New-Item src\research_agent\rag\schemas.py
New-Item src\research_agent\rag\loaders.py

mkdir scripts
New-Item scripts\test_load_docs.py

如果提示文件已存在，不用管。

# 四、写 schemas.py

路径：

F:\ResearchAgent\src\research_agent\rag\schemas.py

复制下面代码：

from typing import Literal, TypedDict


# 三类核心资料类型
SOURCE_TYPE_PAPER = "paper_note"
SOURCE_TYPE_EXPERIMENT = "experiment_doc"
SOURCE_TYPE_DATASET = "dataset_doc"
SOURCE_TYPE_CODE = "code_doc"


SourceType = Literal[
    "paper_note",
    "experiment_doc",
    "dataset_doc",
    "code_doc",
]


class DocumentMetadata(TypedDict, total=False):
    """
    RAG 文档 metadata 规范。

    total=False 表示不是每个字段都必须存在。
    比如 paper 文档可能有 title/topic/year，
    experiment 文档可能有 run_tag/dataset/task。
    """
    source_type: str
    title: str
    path: str

    topic: str
    year: int

    run_tag: str
    dataset: str
    task: str

    annotation_type: str


SOURCE_TYPE_TO_DIR = {
    SOURCE_TYPE_PAPER: "papers",
    SOURCE_TYPE_EXPERIMENT: "experiments",
    SOURCE_TYPE_DATASET: "datasets",
}


DIR_TO_SOURCE_TYPE = {
    "papers": SOURCE_TYPE_PAPER,
    "experiments": SOURCE_TYPE_EXPERIMENT,
    "datasets": SOURCE_TYPE_DATASET,
}


REQUIRED_METADATA_KEYS = [
    "source_type",
    "path",
]


COMMON_OPTIONAL_KEYS = [
    "title",
    "topic",
    "year",
    "run_tag",
    "dataset",
    "task",
    "annotation_type",
]
你要理解什么？

这里先别纠结代码复杂度，你重点理解：

SOURCE_TYPE_PAPER = "paper_note"
SOURCE_TYPE_EXPERIMENT = "experiment_doc"
SOURCE_TYPE_DATASET = "dataset_doc"

这三个 source_type 后面会决定检索范围：

paper_question           → 只查 paper_note
experiment_analysis      → 只查 experiment_doc
dataset_recommendation   → 只查 dataset_doc

所以 metadata 不是装饰品，而是阶段二 Agentic RAG 的关键。

# 五、写 loaders.py

路径：

F:\ResearchAgent\src\research_agent\rag\loaders.py

复制下面代码：

from pathlib import Path
from typing import Dict, List, Optional

from langchain_core.documents import Document

from .schemas import (
    DIR_TO_SOURCE_TYPE,
    REQUIRED_METADATA_KEYS,
)


PROJECT_ROOT = Path(__file__).resolve().parents[3]
DATA_DIR = PROJECT_ROOT / "data"


def read_text_file(path: Path) -> str:
    """
    读取文本文件，统一使用 utf-8 编码。
    """
    return path.read_text(encoding="utf-8")


def normalize_metadata_value(value: str):
    """
    对 metadata 字段做简单类型规范化。

    例如：
    - year: "2026" -> 2026
    - 空字符串 -> 不保留
    """
    value = value.strip()

    if not value:
        return None

    if value.isdigit():
        return int(value)

    return value


def parse_basic_info(markdown_text: str) -> Dict:
    """
    从 markdown 的 “## 基本信息” 部分解析 metadata。

    支持这种格式：

    ## 基本信息

    - source_type: paper_note
    - title: xxx
    - topic: multimodal_bias
    - year: 2026
    - path: data/papers/xxx.md

    返回：
    {
        "source_type": "paper_note",
        "title": "...",
        ...
    }
    """
    metadata = {}

    lines = markdown_text.splitlines()
    in_basic_info = False

    for line in lines:
        stripped = line.strip()

        # 找到基本信息段落
        if stripped == "## 基本信息":
            in_basic_info = True
            continue

        # 如果已经进入基本信息段落，遇到下一个二级标题就结束
        if in_basic_info and stripped.startswith("## ") and stripped != "## 基本信息":
            break

        if not in_basic_info:
            continue

        # 解析 "- key: value"
        if stripped.startswith("- ") and ":" in stripped:
            item = stripped[2:]
            key, value = item.split(":", 1)
            key = key.strip()
            value = normalize_metadata_value(value)

            if value is not None:
                metadata[key] = value

    return metadata


def infer_source_type_from_path(path: Path) -> Optional[str]:
    """
    根据文件所在目录推断 source_type。

    data/papers/*.md      -> paper_note
    data/experiments/*.md -> experiment_doc
    data/datasets/*.md    -> dataset_doc
    """
    parent_name = path.parent.name
    return DIR_TO_SOURCE_TYPE.get(parent_name)


def load_markdown_document(path: Path, project_root: Path = PROJECT_ROOT) -> Document:
    """
    加载单个 markdown 文件，返回 LangChain Document。
    """
    markdown_text = read_text_file(path)
    metadata = parse_basic_info(markdown_text)

    # 如果文档里没有 source_type，就根据目录推断
    if "source_type" not in metadata:
        inferred_source_type = infer_source_type_from_path(path)
        if inferred_source_type:
            metadata["source_type"] = inferred_source_type

    # 如果文档里没有 path，就自动写相对路径
    if "path" not in metadata:
        metadata["path"] = str(path.relative_to(project_root)).replace("\\", "/")

    # 如果文档里没有 title，就用一级标题或文件名
    if "title" not in metadata:
        metadata["title"] = extract_title(markdown_text) or path.stem

    validate_metadata(metadata, path)

    return Document(
        page_content=markdown_text,
        metadata=metadata,
    )


def extract_title(markdown_text: str) -> Optional[str]:
    """
    提取 markdown 第一个一级标题作为 title。
    """
    for line in markdown_text.splitlines():
        stripped = line.strip()
        if stripped.startswith("# "):
            return stripped.replace("# ", "", 1).strip()
    return None


def validate_metadata(metadata: Dict, path: Path) -> None:
    """
    检查必要 metadata 字段是否存在。
    """
    missing_keys = [
        key for key in REQUIRED_METADATA_KEYS
        if key not in metadata or metadata[key] in ["", None]
    ]

    if missing_keys:
        raise ValueError(
            f"Missing required metadata keys {missing_keys} in file: {path}"
        )


def load_markdown_documents(directory: Path) -> List[Document]:
    """
    加载某个目录下所有 markdown 文件。
    """
    if not directory.exists():
        return []

    documents = []

    for path in sorted(directory.glob("*.md")):
        doc = load_markdown_document(path)
        documents.append(doc)

    return documents


def load_all_documents(data_dir: Path = DATA_DIR) -> List[Document]:
    """
    加载 data/papers、data/experiments、data/datasets 下所有 markdown 文档。
    """
    documents = []

    for subdir in ["papers", "experiments", "datasets"]:
        directory = data_dir / subdir
        documents.extend(load_markdown_documents(directory))

    return documents


def summarize_documents(documents: List[Document]) -> Dict[str, int]:
    """
    按 source_type 统计文档数量，方便测试。
    """
    summary = {}

    for doc in documents:
        source_type = doc.metadata.get("source_type", "unknown")
        summary[source_type] = summary.get(source_type, 0) + 1

    return summary


def format_document_preview(doc: Document, max_chars: int = 200) -> str:
    """
    格式化单个 Document 的预览信息。
    """
    content_preview = doc.page_content[:max_chars].replace("\n", " ")

    metadata_lines = [
        f"{key}: {value}"
        for key, value in doc.metadata.items()
    ]

    metadata_text = "\n".join(metadata_lines)

    return f"""
Metadata:
{metadata_text}

Content Preview:
{content_preview}...
""".strip()

In [1]:
from src.research_agent.rag.loaders import load_all_documents

docs = load_all_documents()

doc = docs[0]

print(type(doc))
print(doc.page_content[:300])
print(doc.metadata)

ModuleNotFoundError: No module named 'src'

In [2]:
import os

print("Before:", os.getcwd())

%cd F:\ResearchAgent

print("After:", os.getcwd())

Before: f:\ResearchAgent\notebooks
F:\ResearchAgent
After: F:\ResearchAgent


In [3]:
from src.research_agent.rag.loaders import load_all_documents

docs = load_all_documents()

print(f"Loaded {len(docs)} documents.")

doc = docs[0]

print(type(doc))
print(doc.page_content[:300])
print(doc.metadata)

Loaded 14 documents.
<class 'langchain_core.documents.base.Document'>
# Guardrail-Agnostic Societal Bias Evaluation in LVLMs

## 基本信息

- source_type: paper_note
- title: Guardrail-Agnostic Societal Bias Evaluation in Large Vision-Language Models
- topic: multimodal_bias
- year: 2026
- path: data/papers/guardrail_agnostic.md

## 研究问题

这篇论文关注大型视觉语言模型中的社会偏见评估问题，尤其是当模型存在安
{'source_type': 'paper_note', 'title': 'Guardrail-Agnostic Societal Bias Evaluation in Large Vision-Language Models', 'topic': 'multimodal_bias', 'year': 2026, 'path': 'data/papers/guardrail_agnostic.md'}


In [5]:
print(type(doc))

<class 'langchain_core.documents.base.Document'>
